In [3]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
df=pd.read_csv("BSP.csv")
df.head()

,date,gender,age,address,famsize,Pstatus,M_Edu,F_Edu,M_Job,F_Job,relationship,smoker,tuition_fee,time_friends,ssc_result,hsc_result
0,29/04/2018,M,18,Rural,GT3,Together,3,2,At_home,Farmer,No,No,71672,4,4.22,3.72
1,29/04/2018,F,19,Rural,LE3,Apart,0,4,Other,Health,Yes,No,26085,5,3.47,2.62
2,29/04/2018,F,19,Rural,GT3,Together,0,3,Teacher,Services,No,No,40891,3,3.32,2.56
3,29/04/2018,F,19,Rural,LE3,Apart,2,3,At_home,Business,No,No,50600,2,4.57,4.17
4,29/04/2018,M,17,Rural,GT3,Together,1,1,At_home,Farmer,No,No,62458,2,4.50,3.94


In [4]:
! pip install ydata-profiling 

In [5]:
%pip install ydata-profiling
from ydata_profiling import ProfileReport

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from ydata_profiling import ProfileReport 

In [7]:
profile=ProfileReport(df,title="BSP",explorative=True)
profile.to_file("ydata.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 42.23it/s]


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingRegressor ,StackingRegressor,RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score 
from sklearn.linear_model import LinearRegression,Ridge




In [9]:
df=pd.read_csv("BSP.csv")

df.drop(columns=["date"],inplace=True)
df.columns

Index(['gender', 'age', 'address', 'famsize', 'Pstatus', 'M_Edu', 'F_Edu',
       'M_Job', 'F_Job', 'relationship', 'smoker', 'tuition_fee',
       'time_friends', 'ssc_result', 'hsc_result'],
      dtype='object')

In [10]:
corr_target=df.select_dtypes(include=np.number).corr()['hsc_result'].sort_values(ascending=False)
corr_target

hsc_result      1.000000
ssc_result      0.950178
M_Edu           0.063776
F_Edu           0.054811
tuition_fee     0.038068
age            -0.009857
time_friends   -0.156356
Name: hsc_result, dtype: float64

In [11]:
X=df.drop('hsc_result',axis=1)

y=df['hsc_result']


In [12]:
##Pipeline 
##numerical
num_transformer=Pipeline(
    steps=[
        ('impute',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())


    ]

)



In [13]:
##categorical 
cat_transformer=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(handle_unknown='ignore'))
        
    ]


)

In [14]:
##train test 
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)       

In [15]:
#Ensemble Learning '
#base learner
reg_lr=LinearRegression()
reg_rf=RandomForestRegressor(n_estimators=100,random_state=42)  
reg_gb=GradientBoostingRegressor(n_estimators=100,random_state=42)  


In [16]:
##Voting
voting_reg=VotingRegressor(
    estimators=[('lr',reg_lr),('rf',reg_rf),('gb',reg_gb)]


)

In [17]:
#Stacking 
stacking_reg=StackingRegressor(
    estimators=[('lr',reg_lr),('rf',reg_rf),('gb',reg_gb)],
    final_estimator=Ridge()##the meta learner 
)


In [18]:
model_to_train ={
    'LinearRegression':reg_lr,
    "RamdomForest":reg_rf,
    "GradientBoosting":reg_gb,
    'Voting Ensemble':voting_reg,
    "Stacking Ensemble":stacking_reg    


}


In [19]:
result=[]
for name,model in model_to_train.items():
    pipe=Pipeline(
        [
            ('preprocessor',ColumnTransformer(
                transformers=[
                    ('num',num_transformer,X.select_dtypes(include=np.number).columns),
                    ('cat',cat_transformer,X.select_dtypes(exclude=np.number).columns)
                ]
            )),
            ('model',model)
        ]
    )
    pipe.fit(X_train,y_train)
    y_pred=pipe.predict(X_test)
    r2=r2_score(y_test,y_pred)
    mae=mean_absolute_error(y_test,y_pred)  
    mse=mean_squared_error(y_test,y_pred)
    result.append(
        {
            'Model':name,
            'r2_score':r2,
            'mae':mae,  
            'mse':mse




        }
    )
result_df=pd.DataFrame(result).sort_values(by='r2_score',ascending=False)

In [20]:
result_df

,Model,r2_score,mae,mse
2,GradientBoosting,0.959565,0.098902,0.015155
4,Stacking Ensemble,0.959405,0.098687,0.015215
3,Voting Ensemble,0.957528,0.100838,0.015919
1,RamdomForest,0.950248,0.108201,0.018647
0,LinearRegression,0.945920,0.111376,0.020269


In [21]:
best_model_name=result_df.iloc[0]['Model']
best_model_object=model_to_train[best_model_name]

final_pipe=Pipeline(
    [
        ('preprocessor',ColumnTransformer(
            transformers=[
                ('num',num_transformer,X.select_dtypes(include=np.number).columns),
                ('cat',cat_transformer,X.select_dtypes(exclude=np.number).columns)
            ]
        )),
        ('model',best_model_object)
    ]
)
final_pipe.fit(X_train,y_train)
y_final_pred=final_pipe.predict(X_test)

In [22]:
import seaborn as sns 
plt.figure(figsize=(12,8))
sns.scatterplot(x=y_test,y=y_final_pred,alpha=0.5,color='g')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_72588\3591077729.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
df.head()

,gender,age,address,famsize,Pstatus,M_Edu,F_Edu,M_Job,F_Job,relationship,smoker,tuition_fee,time_friends,ssc_result,hsc_result
0,M,18,Rural,GT3,Together,3,2,At_home,Farmer,No,No,71672,4,4.22,3.72
1,F,19,Rural,LE3,Apart,0,4,Other,Health,Yes,No,26085,5,3.47,2.62
2,F,19,Rural,GT3,Together,0,3,Teacher,Services,No,No,40891,3,3.32,2.56
3,F,19,Rural,LE3,Apart,2,3,At_home,Business,No,No,50600,2,4.57,4.17
4,M,17,Rural,GT3,Together,1,1,At_home,Farmer,No,No,62458,2,4.50,3.94


In [24]:
from sklearn.model_selection import cross_val_score
rf_pipeline=Pipeline(
    [
        ('preprocessor',ColumnTransformer(
            transformers=[
                ('num',num_transformer,X.select_dtypes(include=np.number).columns),
                ('cat',cat_transformer,X.select_dtypes(exclude=np.number).columns)
            ]
        )),
        ('model',RandomForestRegressor(n_estimators=100,random_state=42))
    ]
)

In [25]:


cv_scores=cross_val_score(rf_pipeline,X_train,y_train,cv=10,scoring='neg_mean_squared_error')
cv_rmse=np.sqrt(-cv_scores)
print(cv_rmse)


[0.1422733  0.13062705 0.13229842 0.14680644 0.14795956 0.14658055
 0.15397954 0.13899758 0.12200084 0.13500019]


In [26]:
print(cv_rmse.std())

0.00921689795310348


In [27]:
##randomized searchcv 
from scipy.stats import randint 
my_dist=randint(1,11)
print(my_dist.rvs(size=6))

[7 2 3 3 3 4]


In [28]:
param_dist={
    'model__n_estimators':randint(100,500),
    'model__max_depth':[None,10,20],
    'model__min_samples_split':randint(1,10)

}

from sklearn.model_selection import RandomizedSearchCV
random_search=RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    n_jobs=-1,
    scoring='neg_mean_squared_error',
    verbose=2,
    random_state=42





)
random_search.fit(X_train,y_train)
print("Best Hyperparameters:",random_search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
20 fits failed out of a total of 250.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\L

Best Hyperparameters: {'model__max_depth': 10, 'model__min_samples_split': 4, 'model__n_estimators': 369}


In [29]:
import pickle 
from sklearn.linear_model import LinearRegression
X_train_lr=[[1],[2],[3],[4],[5]]
y_train_lr=[10,20,30,40,50]
model=LinearRegression()
model.fit(X_train,y_train)
model.predict([[10]])[0]

ValueError: could not convert string to float: 'M'

In [ ]:
filename="lr.pkl"
with open(filename,"wb")as file :
    pickle.dump(model,file)

In [ ]:
with open("lr.pkl","rb") as file:
    loaded_model=pickle.load(file)

In [ ]:
loaded_model.predict([[7]])

array([70.])

In [ ]:
##saving random forest 
filename="random_forest_model.pkl"
with open(filename,"wb")as file:
    pickle.dump(random_search,file)
    

In [ ]:
with open("random_forest_model.pkl","rb")as file:
    loaded_rf_model=pickle.load(file)
    print(loaded_rf_model.predict(X_test))


[3.24813808 3.63936138 3.69632315 3.93415944 4.1076736  3.27072695
 4.09958465 2.50256598 2.10633699 3.2701777  3.57431522 2.92584264
 3.19929278 4.51298167 3.01386118 2.63695208 3.61628147 3.36786373
 2.87122621 2.88735632 2.74107849 3.18588724 2.89945616 3.22534203
 2.05461314 2.64539077 3.21802007 4.03430123 3.1203105  4.08801523
 2.45744916 3.83485003 4.50888024 2.56348902 3.20317019 3.21390665
 2.74812963 3.57604872 2.45020837 2.60565241 2.98895371 2.97619076
 3.66365835 2.76651367 2.32610757 3.01739181 2.61563775 2.70441272
 4.1070183  3.22731134 3.00102886 3.40829784 4.236945   3.53883388
 2.99426729 3.23744524 3.25401752 2.78525418 3.71359346 2.7038779
 3.03148646 2.65733352 2.5149602  4.43089257 2.58082166 3.21989986
 3.86168574 2.73406531 3.04757297 3.23642417 3.01372702 2.62950665
 2.70293356 3.43821656 3.04831729 2.78659414 3.54885181 3.00970432
 2.51008511 4.22198992 3.32861618 2.61590302 3.22596768 3.09251686
 2.37853671 3.37573203 3.69604185 2.71164173 2.82403445 4.16743

In [ ]:
! pip install mlflow 


In [ ]:
import mlflow 

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("test_run")
with mlflow.start_run(run_name='dummy_test'):
    mlflow.log_metric("Accuracy",0.95)
    mlflow.log_metric("loss",0.05)
    mlflow.log_param("model","fake_modelv1")
    mlflow.log_param("Learning Rate",0.01)

2026/02/08 01:11:13 INFO mlflow.tracking.fluent: Experiment with name 'test_run' does not exist. Creating a new experiment.


In [31]:
##RFMLFLOW
import mlflow 
import mlflow.sklearn 
import numpy as np 
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error 
from sklearn.ensemble import RandomForestRegressor  
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
mlflow.set_experiment("RFBSP")
my_params={
    'n_estimators':100,
    'max_depth':10,
    'random_state':42
}
simple_rf_pipeline=Pipeline(
    [
        ('preprocessor',ColumnTransformer(
            transformers=[
                ('num',num_transformer,X.select_dtypes(include=np.number).columns),
                ('cat',cat_transformer,X.select_dtypes(exclude=np.number).columns)
            ]
        )),
        ('model',RandomForestRegressor(**my_params))
    ]
)
with mlflow.start_run(run_name='singlerf'):
 mlflow.log_params(my_params)
 mlflow.log_param("model","RandomForestRegressor")
 simple_rf_pipeline.fit(X_train,y_train)
 y_train_pred=simple_rf_pipeline.predict(X_train)
 train_rmse=np.sqrt(mean_squared_error(y_train,y_train_pred))
 mlflow.log_metric("train_rmse",train_rmse)
 y_test_pred=simple_rf_pipeline.predict(X_test)
 test_rmse=np.sqrt(mean_squared_error(y_test,y_test_pred))
 mlflow.log_metric("test_rmse",test_rmse)




2026/02/08 01:35:12 INFO mlflow.tracking.fluent: Experiment with name 'RFBSP' does not exist. Creating a new experiment.
